# 文档分类工程：从标签体系到可拒识的批量推理

文档分类不是“把一段文字丢给模型并取最大概率”。一个可上线系统至少要同时管理：业务动作、标签定义、数据切分、特征与模型版本、阈值、拒识、漂移、批量接口和回归测试。

本 Notebook 用一个**完全自带、可重复生成的受控中文工单集**，实现 TF-IDF + Logistic Regression，并与 Dummy / Naive Bayes 基线比较。数据是为了暴露工程步骤而设计的，样本量小、模板痕迹强；这里出现的高分不能外推到真实业务，也不能替代人工标注的时间外测试集。

## 学习目标

1. 把单标签、多标签与层级分类写成可检查的 label schema；
2. 避免时间泄漏、实体泄漏和预处理泄漏；
3. 建立可比较的 TF-IDF 基线，并正确解释 macro/micro F1；
4. 处理类别不平衡、概率校准、阈值与 OOD/拒识；
5. 固化模型/词表版本和批量推理契约，并用断言守住关键不变量。


## 1. 先定义业务问题和 label schema

分类输出必须对应一个动作。例如 `technical.network` 路由到网络支持队列，`account.security` 触发更严格的身份核验。仅有中文标签名还不够，生产 schema 至少应记录稳定 ID、展示名、父节点、定义、正例/反例、互斥规则、负责人、版本与弃用策略。

三类常见任务不要混在一起：

- **单标签**：每个工单恰好进入一个主队列，通常用 softmax/多项分类；
- **多标签**：同一工单可同时有 `退款`、`紧急`、`疑似欺诈`，每个标签独立决策并拥有自己的阈值；
- **层级分类**：`billing.invoice` 的父类是 `billing`。可以先父后子，也可以扁平预测后做层级一致性校验。若输出子类却未输出父类，必须补父类或拒绝。

本例训练目标是五选一的 `label_l2`；`label_l1` 和 `labels_multi` 用来演示层级与多标签契约，不把不同任务的指标混为一谈。


In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

from collections import Counter  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from datetime import datetime  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。
import unicodedata  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import pandas as pd  # 导入本单元所需的依赖。
import sklearn  # 导入本单元所需的依赖。
from sklearn.base import clone  # 导入本单元所需的依赖。
from sklearn.calibration import CalibratedClassifierCV  # 导入本单元所需的依赖。
from sklearn.dummy import DummyClassifier  # 导入本单元所需的依赖。
from sklearn.feature_extraction.text import TfidfVectorizer  # 导入本单元所需的依赖。
from sklearn.linear_model import LogisticRegression  # 导入本单元所需的依赖。
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score)  # 导入本单元所需的依赖。
from sklearn.model_selection import GroupShuffleSplit  # 导入本单元所需的依赖。
from sklearn.naive_bayes import MultinomialNB  # 导入本单元所需的依赖。
from sklearn.pipeline import Pipeline, make_pipeline  # 导入本单元所需的依赖。
from sklearn.preprocessing import MultiLabelBinarizer  # 导入本单元所需的依赖。

RANDOM_STATE = 42  # 计算并保存当前步骤的中间状态。
LABEL_SCHEMA = {  # 计算并保存当前步骤的中间状态。
    "billing.invoice": {"name": "账单与发票", "parent": "billing"},  # 执行当前语句以推进本节示例。
    "billing.refund": {"name": "退款", "parent": "billing"},  # 执行当前语句以推进本节示例。
    "technical.network": {"name": "网络故障", "parent": "technical"},  # 执行当前语句以推进本节示例。
    "account.security": {"name": "账号安全", "parent": "account"},  # 执行当前语句以推进本节示例。
    "logistics.delivery": {"name": "物流配送", "parent": "logistics"},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
PHRASES = {  # 计算并保存当前步骤的中间状态。
    "billing.invoice": ["电子发票一直没有开出", "发票抬头需要修改", "账单金额与订单不一致", "如何下载增值税发票", "开票税号填写错误", "月度账单无法查看"],  # 执行当前语句以推进本节示例。
    "billing.refund": ["取消订单后退款未到账", "申请原路退回付款", "退款进度一直没有更新", "重复扣款需要退回", "审核通过但余额未恢复", "想撤销服务并办理退款"],  # 执行当前语句以推进本节示例。
    "technical.network": ["客户端连接服务器超时", "公司网络频繁断线", "接口返回网关错误", "域名解析后仍无法访问", "上传文件时连接被重置", "专线延迟突然升高"],  # 执行当前语句以推进本节示例。
    "account.security": ["发现账号在异地登录", "密码重置链接已经失效", "怀疑访问密钥发生泄露", "手机丢失无法完成二次验证", "账户被陌生设备登录", "需要冻结高风险账号"],  # 执行当前语句以推进本节示例。
    "logistics.delivery": ["包裹显示签收但没有收到", "物流轨迹多天没有更新", "收货地址需要临时修改", "快递外包装严重破损", "订单一直等待仓库发货", "配送时间超过了承诺日期"],  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

records = []  # 计算并保存当前步骤的中间状态。
channels = ["网页", "电话", "邮件", "App"]  # 计算并保存当前步骤的中间状态。
products = ["标准版", "专业版", "企业版"]  # 计算并保存当前步骤的中间状态。
time_segments = ["年初批次", "春季批次", "夏季批次", "秋季批次"]  # 计算并保存当前步骤的中间状态。
for label_l2, phrases in PHRASES.items():  # 遍历输入元素以累积或检查结果。
    for step in range(24):  # 遍历输入元素以累积或检查结果。
        urgent = step % 7 == 0  # 计算并保存当前步骤的中间状态。
        text = (  # 计算并保存当前步骤的中间状态。
            f"{channels[step % len(channels)]}工单：{phrases[step % len(phrases)]}；"  # 执行当前语句以推进本节示例。
            f"{time_segments[step // 6]}，使用{products[step % len(products)]}，"  # 执行当前语句以推进本节示例。
            f"{'非常紧急，请优先处理' if urgent else '请客服协助处理'}。"  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        records.append({  # 执行当前语句以推进本节示例。
            "doc_id": f"D-{len(records):04d}",  # 执行当前语句以推进本节示例。
            # 每个时间步使用独立客户；同一客户可有多个标签工单，但不会跨时间 split。
            "customer_id": f"C-{step:02d}",  # 执行当前语句以推进本节示例。
            "created_at": pd.Timestamp("2025-01-01") + pd.Timedelta(days=7 * step),  # 计算并保存当前步骤的中间状态。
            "text": text,  # 执行当前语句以推进本节示例。
            "label_l2": label_l2,  # 执行当前语句以推进本节示例。
            "label_l1": LABEL_SCHEMA[label_l2]["parent"],  # 执行当前语句以推进本节示例。
            "labels_multi": (label_l2, "priority.urgent") if urgent else (label_l2,),  # 执行当前语句以推进本节示例。
            "sequence": step,  # 执行当前语句以推进本节示例。
        })  # 执行当前语句以推进本节示例。

df = pd.DataFrame(records).sort_values(["created_at", "doc_id"]).reset_index(drop=True)  # 计算并保存当前步骤的中间状态。
print(f"sklearn={sklearn.__version__}, 样本数={len(df)}, 类别分布={dict(df.label_l2.value_counts())}")  # 计算并保存当前步骤的中间状态。
df.head(3)  # 执行当前语句以推进本节示例。


## 2. 切分：时间、实体和预处理泄漏

随机切分只在样本近似独立同分布时合理。真实工单通常存在三类隐蔽泄漏：

1. **时间泄漏**：用未来模板、未来标签规则或未来词表预测过去；
2. **实体泄漏**：同一客户、合同或文档修订版同时进入训练和测试，模型记住了实体措辞；
3. **预处理泄漏**：先在全量数据上拟合 TF-IDF、去重规则、特征选择或校准器，再切分。

下面保留严格的 `train -> validation -> test` 时间顺序，并让三个时间段的 `customer_id` 不交叉。合成文本加入所有标签共享的时间段措辞，用来消除跨 split 的精确模板重复；该措辞不携带标签信息。validation 用于模型/类别权重比较、错误分析和阈值选择，test 在方案冻结后只做一次最终报告。另用 GroupShuffleSplit 展示一般化的“按客户留出”检查。即便时间、客户与精确文本都隔离，这个模板合成集仍不是对真实语言泛化能力的估计。


In [ ]:
df["split"] = np.select(  # 计算并保存当前步骤的中间状态。
    [df["sequence"] < 14, df["sequence"] < 18],  # 执行当前语句以推进本节示例。
    ["train", "validation"],  # 执行当前语句以推进本节示例。
    default="test",  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
train_df = df[df.split == "train"].copy()  # 计算并保存当前步骤的中间状态。
valid_df = df[df.split == "validation"].copy()  # 计算并保存当前步骤的中间状态。
test_df = df[df.split == "test"].copy()  # 计算并保存当前步骤的中间状态。

group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)  # 计算并保存当前步骤的中间状态。
entity_train_idx, entity_test_idx = next(group_splitter.split(df, groups=df.customer_id))  # 计算并保存当前步骤的中间状态。
entity_train = df.iloc[entity_train_idx]  # 计算并保存当前步骤的中间状态。
entity_test = df.iloc[entity_test_idx]  # 计算并保存当前步骤的中间状态。

assert train_df.created_at.max() < valid_df.created_at.min() < test_df.created_at.min()  # 用受控断言验证关键不变量。
time_split_customer_sets = [set(frame.customer_id) for frame in (train_df, valid_df, test_df)]  # 计算并保存当前步骤的中间状态。
assert all(left.isdisjoint(right) for i, left in enumerate(time_split_customer_sets) for right in time_split_customer_sets[i + 1:])  # 用受控断言验证关键不变量。
assert set(entity_train.customer_id).isdisjoint(entity_test.customer_id)  # 用受控断言验证关键不变量。
print(pd.crosstab(df.split, df.label_l2))  # 执行当前语句以推进本节示例。
print("实体外测试客户：", sorted(entity_test.customer_id.unique()))  # 执行当前语句以推进本节示例。


## 3. 清洗要可版本化，不能把业务信号洗掉

清洗通常包括 Unicode 规范化、控制字符处理、空白统一和确定性的 PII 脱敏。不要盲目删除数字、标点、否定词或大小写：`退款到账` 与 `退款未到账`、错误码 `E100` 与 `E101` 的差异可能正是标签证据。

去重也要在切分前按“文档族”分组，或在训练集内部完成；如果先把近重复样本随机分散到两侧，测试分数会虚高。所有清洗函数、规则表和词典都应进入 `preprocess_version`。


In [ ]:
PREPROCESS_VERSION = "clean-v1"  # 计算并保存当前步骤的中间状态。

def clean_text(text: str) -> str:  # 定义本节可复用的核心函数。
    if not isinstance(text, str):  # 按当前条件选择后续控制路径。
        raise TypeError("text 必须是字符串")  # 遇到非法合同立即显式失败。
    text = unicodedata.normalize("NFKC", text)  # 计算并保存当前步骤的中间状态。
    text = re.sub(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f]", " ", text)  # 计算并保存当前步骤的中间状态。
    text = re.sub(r"\b1[3-9]\d{9}\b", "<PHONE>", text)  # 计算并保存当前步骤的中间状态。
    return re.sub(r"\s+", " ", text).strip()  # 返回当前分支计算出的结果。

for frame in (train_df, valid_df, test_df):  # 遍历输入元素以累积或检查结果。
    frame["clean_text"] = frame.text.map(clean_text)  # 计算并保存当前步骤的中间状态。

clean_text_sets = [set(frame.clean_text) for frame in (train_df, valid_df, test_df)]  # 计算并保存当前步骤的中间状态。
assert all(frame.clean_text.is_unique for frame in (train_df, valid_df, test_df))  # 用受控断言验证关键不变量。
assert all(left.isdisjoint(right) for i, left in enumerate(clean_text_sets) for right in clean_text_sets[i + 1:])  # 用受控断言验证关键不变量。

mlb = MultiLabelBinarizer()  # 计算并保存当前步骤的中间状态。
multi_hot = mlb.fit_transform(train_df.labels_multi)  # 计算并保存当前步骤的中间状态。
assert all(LABEL_SCHEMA[row.label_l2]["parent"] == row.label_l1 for row in df.itertuples())  # 用受控断言验证关键不变量。
print("多标签列：", list(mlb.classes_), "shape=", multi_hot.shape)  # 计算并保存当前步骤的中间状态。
print("清洗示例：", clean_text("账号  异常，联系电话 13800138000"))  # 执行当前语句以推进本节示例。


## 4. TF-IDF baseline 与可比较实验

TF-IDF 对词/字符片段加权。这里选中文字符 2–4 gram，避免额外分词依赖，也能捕捉“退款未到”“异地登录”等局部模式。它不理解语义，面对同义改写和长程组合会失败，但训练快、可解释，是排查数据管道和标签质量的重要基线。

比较实验必须共享同一切分、同一文本列和同一指标。`DummyClassifier` 给出多数类下限，MultinomialNB 是稀疏文本的低成本基线，Logistic Regression 提供线性决策与概率接口。向量器放进 Pipeline，确保只在训练折上 `fit`。


In [ ]:
def make_tfidf() -> TfidfVectorizer:  # 定义本节可复用的核心函数。
    return TfidfVectorizer(  # 返回当前分支计算出的结果。
        analyzer="char", ngram_range=(2, 4), min_df=1,  # 计算并保存当前步骤的中间状态。
        sublinear_tf=True, max_features=4000, dtype=np.float64,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。

model_specs = {  # 计算并保存当前步骤的中间状态。
    "Dummy-most-frequent": DummyClassifier(strategy="most_frequent"),  # 计算并保存当前步骤的中间状态。
    "MultinomialNB": MultinomialNB(alpha=0.5),  # 计算并保存当前步骤的中间状态。
    "LogisticRegression": LogisticRegression(C=3.0, max_iter=1000, random_state=RANDOM_STATE),  # 计算并保存当前步骤的中间状态。
}  # 执行当前语句以推进本节示例。
fitted_models, score_rows = {}, []  # 计算并保存当前步骤的中间状态。
for name, estimator in model_specs.items():  # 遍历输入元素以累积或检查结果。
    pipeline = Pipeline([("tfidf", make_tfidf()), ("classifier", clone(estimator))])  # 计算并保存当前步骤的中间状态。
    pipeline.fit(train_df.clean_text, train_df.label_l2)  # 执行当前语句以推进本节示例。
    prediction = pipeline.predict(valid_df.clean_text)  # 计算并保存当前步骤的中间状态。
    fitted_models[name] = pipeline  # 计算并保存当前步骤的中间状态。
    score_rows.append({  # 执行当前语句以推进本节示例。
        "model": name,  # 执行当前语句以推进本节示例。
        "accuracy": accuracy_score(valid_df.label_l2, prediction),  # 执行当前语句以推进本节示例。
        "macro_f1": f1_score(valid_df.label_l2, prediction, average="macro", zero_division=0),  # 计算并保存当前步骤的中间状态。
        "micro_f1": f1_score(valid_df.label_l2, prediction, average="micro", zero_division=0),  # 计算并保存当前步骤的中间状态。
    })  # 执行当前语句以推进本节示例。

scores = pd.DataFrame(score_rows).sort_values("macro_f1", ascending=False)  # 计算并保存当前步骤的中间状态。
scores.round(3)  # 执行当前语句以推进本节示例。


## 5. macro/micro F1、混淆矩阵与错误分析

单标签多分类中，micro F1 会被大类主导且等于 accuracy；macro F1 先算每类 F1 再平均，小类失败会明显拉低它。生产报告还应给每类 precision/recall、支持样本数和业务成本，而不是只给一个平均数。

混淆矩阵的行是真值、列是预测。错误分析要回到原文、标签定义、模型置信度和数据来源，按“标签歧义、文本信息不足、清洗错误、领域外、时间漂移”等根因分桶。不能看完测试错误后直接改规则再汇报同一个测试集分数；那会把测试集变成训练集。


In [ ]:
LABELS = sorted(LABEL_SCHEMA)  # 计算并保存当前步骤的中间状态。
logistic_model = fitted_models["LogisticRegression"]  # 计算并保存当前步骤的中间状态。
analysis_pred = logistic_model.predict(valid_df.clean_text)  # 计算并保存当前步骤的中间状态。
analysis_prob = logistic_model.predict_proba(valid_df.clean_text)  # 计算并保存当前步骤的中间状态。
cm = pd.DataFrame(  # 计算并保存当前步骤的中间状态。
    confusion_matrix(valid_df.label_l2, analysis_pred, labels=LABELS),  # 计算并保存当前步骤的中间状态。
    index=[f"true:{label}" for label in LABELS],  # 计算并保存当前步骤的中间状态。
    columns=[f"pred:{label}" for label in LABELS],  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
error_table = valid_df[["doc_id", "created_at", "clean_text", "label_l2"]].copy()  # 计算并保存当前步骤的中间状态。
error_table["prediction"] = analysis_pred  # 计算并保存当前步骤的中间状态。
error_table["confidence"] = analysis_prob.max(axis=1)  # 计算并保存当前步骤的中间状态。
error_table["is_error"] = error_table.label_l2 != error_table.prediction  # 计算并保存当前步骤的中间状态。
print("混淆矩阵：")  # 执行当前语句以推进本节示例。
display(cm)  # 执行当前语句以推进本节示例。
print("优先复核：错误在前，其次低置信度")  # 执行当前语句以推进本节示例。
error_table.sort_values(["is_error", "confidence"], ascending=[False, True]).head(8)  # 计算并保存当前步骤的中间状态。


## 6. 类别不平衡：先改评估，再谈采样和权重

多数类准确率会掩盖少数类完全不可用。先保留真实测试分布并报告 macro F1/每类召回，再在训练侧比较 `class_weight='balanced'`、过采样、欠采样或难例采样。重复过采样会过拟合重复文本；权重过大则可能牺牲常见类并放大错标。阈值和损失权重都应由业务误判成本与 validation 决定。

下面人为把物流类训练样本压到 3 条，仅演示对照实验。结果仍受模板数据支配，不代表权重在真实数据一定更好。


In [ ]:
minority = train_df[train_df.label_l2 == "logistics.delivery"].head(3)  # 计算并保存当前步骤的中间状态。
imbalanced_train = pd.concat([train_df[train_df.label_l2 != "logistics.delivery"], minority])  # 计算并保存当前步骤的中间状态。
imbalance_rows = []  # 计算并保存当前步骤的中间状态。
for setting, class_weight in [("unweighted", None), ("balanced", "balanced")]:  # 遍历输入元素以累积或检查结果。
    model = make_pipeline(  # 计算并保存当前步骤的中间状态。
        make_tfidf(),  # 执行当前语句以推进本节示例。
        LogisticRegression(C=3.0, class_weight=class_weight, max_iter=1000, random_state=RANDOM_STATE),  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    model.fit(imbalanced_train.clean_text, imbalanced_train.label_l2)  # 执行当前语句以推进本节示例。
    pred = model.predict(valid_df.clean_text)  # 计算并保存当前步骤的中间状态。
    imbalance_rows.append({  # 执行当前语句以推进本节示例。
        "setting": setting,  # 执行当前语句以推进本节示例。
        "macro_f1": f1_score(valid_df.label_l2, pred, average="macro", zero_division=0),  # 计算并保存当前步骤的中间状态。
        "logistics_recall": np.mean(pred[valid_df.label_l2.to_numpy() == "logistics.delivery"] == "logistics.delivery"),  # 计算并保存当前步骤的中间状态。
    })  # 执行当前语句以推进本节示例。
pd.DataFrame(imbalance_rows).round(3)  # 执行当前语句以推进本节示例。


## 7. 概率校准与拒识阈值

分类器给出的 `0.9` 不一定意味着约 90% 的同类样本正确。校准评估的是“置信度是否对应经验正确率”，与排序能力不是一回事。这里用 `CalibratedClassifierCV(method='sigmoid')`：每个内部校准器只看基模型未训练过的折，避免直接拿训练内预测做校准。

拒识阈值也不是固定的 `0.5`。我们在独立 validation 上，约束覆盖率至少 70%，再选择接受样本准确率最高的阈值；最后只在 test 上报告覆盖率和选择性准确率。真实系统应按类别或业务成本设阈值，并同时画 risk–coverage 曲线。


In [ ]:
calibration_base = make_pipeline(  # 计算并保存当前步骤的中间状态。
    make_tfidf(),  # 执行当前语句以推进本节示例。
    LogisticRegression(C=3.0, max_iter=1000, random_state=RANDOM_STATE),  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
calibrated_model = CalibratedClassifierCV(calibration_base, method="sigmoid", cv=3, ensemble=True)  # 计算并保存当前步骤的中间状态。
calibrated_model.fit(train_df.clean_text, train_df.label_l2)  # 执行当前语句以推进本节示例。

def expected_calibration_error(y_true, probabilities, classes, n_bins=5):  # 定义本节可复用的核心函数。
    confidence = probabilities.max(axis=1)  # 计算并保存当前步骤的中间状态。
    prediction = classes[probabilities.argmax(axis=1)]  # 计算并保存当前步骤的中间状态。
    correctness = (prediction == np.asarray(y_true)).astype(float)  # 计算并保存当前步骤的中间状态。
    edges, ece = np.linspace(0.0, 1.0, n_bins + 1), 0.0  # 计算并保存当前步骤的中间状态。
    for left, right in zip(edges[:-1], edges[1:]):  # 遍历输入元素以累积或检查结果。
        mask = (confidence > left) & (confidence <= right)  # 计算并保存当前步骤的中间状态。
        if mask.any():  # 按当前条件选择后续控制路径。
            ece += mask.mean() * abs(correctness[mask].mean() - confidence[mask].mean())  # 计算并保存当前步骤的中间状态。
    return float(ece)  # 返回当前分支计算出的结果。

valid_prob = calibrated_model.predict_proba(valid_df.clean_text)  # 计算并保存当前步骤的中间状态。
valid_pred = calibrated_model.classes_[valid_prob.argmax(axis=1)]  # 计算并保存当前步骤的中间状态。
threshold_rows = []  # 计算并保存当前步骤的中间状态。
for threshold in np.linspace(0.35, 0.90, 12):  # 遍历输入元素以累积或检查结果。
    accepted = valid_prob.max(axis=1) >= threshold  # 计算并保存当前步骤的中间状态。
    threshold_rows.append({  # 执行当前语句以推进本节示例。
        "threshold": float(threshold),  # 执行当前语句以推进本节示例。
        "coverage": float(accepted.mean()),  # 执行当前语句以推进本节示例。
        "accepted_accuracy": float(np.mean(valid_pred[accepted] == valid_df.label_l2.to_numpy()[accepted])) if accepted.any() else 0.0,  # 计算并保存当前步骤的中间状态。
    })  # 执行当前语句以推进本节示例。
eligible = [row for row in threshold_rows if row["coverage"] >= 0.70]  # 计算并保存当前步骤的中间状态。
selected_threshold = max(eligible, key=lambda row: (row["accepted_accuracy"], row["threshold"]))["threshold"]  # 计算并保存当前步骤的中间状态。
test_cal_prob = calibrated_model.predict_proba(test_df.clean_text)  # 计算并保存当前步骤的中间状态。
test_cal_pred = calibrated_model.classes_[test_cal_prob.argmax(axis=1)]  # 计算并保存当前步骤的中间状态。
test_accepted = test_cal_prob.max(axis=1) >= selected_threshold  # 计算并保存当前步骤的中间状态。
print("validation 阈值表：")  # 执行当前语句以推进本节示例。
display(pd.DataFrame(threshold_rows).round(3))  # 执行当前语句以推进本节示例。
print({  # 执行当前语句以推进本节示例。
    "selected_threshold": round(selected_threshold, 3),  # 执行当前语句以推进本节示例。
    "test_coverage": round(float(test_accepted.mean()), 3),  # 执行当前语句以推进本节示例。
    "test_selective_accuracy": round(float(np.mean(test_cal_pred[test_accepted] == test_df.label_l2.to_numpy()[test_accepted])), 3),  # 计算并保存当前步骤的中间状态。
    "test_ece": round(expected_calibration_error(test_df.label_l2, test_cal_prob, calibrated_model.classes_), 3),  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。


## 8. OOD、拒识与漂移监控

最大类别概率低只能发现一部分领域外输入；神经网络和线性模型都可能对陌生文本过度自信。本例再加一个“字符 n-gram 词表覆盖率”信号：低置信度或低覆盖率任一成立就拒识。它只是便宜的守门基线，不是可靠的 OOD 定理；生产中还可组合 embedding 距离、能量分数、专门 OOD 集和人工兜底。

漂移至少分三层监控：输入长度/语言/词表覆盖等协变量漂移，预测标签与置信度漂移，以及有延迟真值后的性能漂移。只有输入 PSI 上升不能证明模型变差，但应触发抽样标注和根因分析。


In [ ]:
def lexical_coverage(text: str, fitted_pipeline: Pipeline) -> float:  # 定义本节可复用的核心函数。
    vectorizer = fitted_pipeline.named_steps["tfidf"]  # 计算并保存当前步骤的中间状态。
    grams = vectorizer.build_analyzer()(clean_text(text))  # 计算并保存当前步骤的中间状态。
    return float(np.mean([gram in vectorizer.vocabulary_ for gram in grams])) if grams else 0.0  # 返回当前分支计算出的结果。

def ood_decision(text: str) -> dict:  # 定义本节可复用的核心函数。
    probabilities = calibrated_model.predict_proba([clean_text(text)])[0]  # 计算并保存当前步骤的中间状态。
    confidence = float(probabilities.max())  # 计算并保存当前步骤的中间状态。
    coverage = lexical_coverage(text, logistic_model)  # 计算并保存当前步骤的中间状态。
    rejected = confidence < selected_threshold or coverage < 0.15  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "text": text, "confidence": confidence, "lexical_coverage": coverage,  # 执行当前语句以推进本节示例。
        "decision": "__REJECT__" if rejected else calibrated_model.classes_[probabilities.argmax()],  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

ood_examples = ["发票税号写错了怎么修改", "给我推荐一道番茄炒蛋", "量子引力的路径积分如何求解"]  # 计算并保存当前步骤的中间状态。
display(pd.DataFrame([ood_decision(text) for text in ood_examples]).round(3))  # 执行当前语句以推进本节示例。

def population_stability_index(reference, current, bins=5, epsilon=1e-6):  # 定义本节可复用的核心函数。
    boundaries = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))  # 计算并保存当前步骤的中间状态。
    boundaries[0], boundaries[-1] = -np.inf, np.inf  # 计算并保存当前步骤的中间状态。
    ref_hist = np.histogram(reference, bins=boundaries)[0] / len(reference)  # 计算并保存当前步骤的中间状态。
    cur_hist = np.histogram(current, bins=boundaries)[0] / len(current)  # 计算并保存当前步骤的中间状态。
    ref_hist, cur_hist = np.clip(ref_hist, epsilon, None), np.clip(cur_hist, epsilon, None)  # 计算并保存当前步骤的中间状态。
    return float(np.sum((cur_hist - ref_hist) * np.log(cur_hist / ref_hist)))  # 返回当前分支计算出的结果。

reference_lengths = train_df.clean_text.str.len().to_numpy()  # 计算并保存当前步骤的中间状态。
current_texts = ["系统不工作", "UNKNOWN COMMAND 9931", "这是一个突然变得非常非常长且和历史工单措辞完全不同的新渠道投诉文本"] * 8  # 计算并保存当前步骤的中间状态。
current_lengths = np.array([len(text) for text in current_texts])  # 计算并保存当前步骤的中间状态。
print({  # 执行当前语句以推进本节示例。
    "length_PSI": round(population_stability_index(reference_lengths, current_lengths), 3),  # 执行当前语句以推进本节示例。
    "current_mean_vocab_coverage": round(float(np.mean([lexical_coverage(t, logistic_model) for t in current_texts])), 3),  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。


## 9. 模型/词表版本与批量推理契约

线上结果必须能回答“哪份数据、标签、清洗、词表、模型和阈值产生了它”。至少发布：`label_schema_version`、`preprocess_version`、向量器参数与词表 hash、模型制品 hash、训练时间窗、依赖版本、校准方法、阈值和评估报告。`CalibratedClassifierCV(ensemble=True)` 内部有多个折模型，每个折都拥有实际参与预测的 TF-IDF 词表；不能拿另一个未校准基线的单一词表 hash 冒充预测器特征空间。

批量接口应规定必填字段、最大 batch、顺序、逐条错误还是整批失败、超时、幂等键和输出 schema。下面选择：输入验证失败整批拒绝；bundle 只服务绑定 tenant；`event_time` 必须是带时区的 ISO 8601；输出与输入同序并携带 bundle fingerprint。生产日志应记录 request_id 和摘要，不默认记录可能含 PII 的全文。


In [ ]:
def tfidf_feature_space_hash(vectorizer: TfidfVectorizer) -> str:  # 定义本节可复用的核心函数。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "analyzer": vectorizer.analyzer,  # 执行当前语句以推进本节示例。
        "ngram_range": list(vectorizer.ngram_range),  # 执行当前语句以推进本节示例。
        "min_df": vectorizer.min_df, "max_df": vectorizer.max_df,  # 执行当前语句以推进本节示例。
        "max_features": vectorizer.max_features, "sublinear_tf": vectorizer.sublinear_tf,  # 执行当前语句以推进本节示例。
        "norm": vectorizer.norm, "use_idf": vectorizer.use_idf,  # 执行当前语句以推进本节示例。
        "vocabulary": sorted((token, int(index)) for token, index in vectorizer.vocabulary_.items()),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    payload = json.dumps(manifest, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()  # 返回当前分支计算出的结果。

feature_space_hashes = tuple(  # 计算并保存当前步骤的中间状态。
    tfidf_feature_space_hash(next(step for step in fold.estimator.named_steps.values() if isinstance(step, TfidfVectorizer)))  # 执行当前语句以推进本节示例。
    for fold in calibrated_model.calibrated_classifiers_  # 遍历输入元素以累积或检查结果。
)  # 执行当前语句以推进本节示例。
coverage_feature_space_hash = tfidf_feature_space_hash(logistic_model.named_steps["tfidf"])  # 计算并保存当前步骤的中间状态。
bundle_manifest = {  # 计算并保存当前步骤的中间状态。
    "model_version": "docclf-lr-2026-07-27",  # 执行当前语句以推进本节示例。
    "label_schema_version": "ticket-labels-v1",  # 执行当前语句以推进本节示例。
    "preprocess_version": PREPROCESS_VERSION,  # 执行当前语句以推进本节示例。
    "serving_tenant": "tenant-a",  # 执行当前语句以推进本节示例。
    "sklearn_version": sklearn.__version__,  # 执行当前语句以推进本节示例。
    "calibration_method": calibrated_model.method,  # 执行当前语句以推进本节示例。
    "classes": [str(label) for label in calibrated_model.classes_],  # 执行当前语句以推进本节示例。
    "threshold": float(selected_threshold),  # 执行当前语句以推进本节示例。
    "feature_space_hashes": feature_space_hashes,  # 执行当前语句以推进本节示例。
    "coverage_feature_space_hash": coverage_feature_space_hash,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
bundle_fingerprint = hashlib.sha256(  # 计算并保存当前步骤的中间状态。
    json.dumps(bundle_manifest, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")  # 计算并保存当前步骤的中间状态。
).hexdigest()  # 执行当前语句以推进本节示例。
MODEL_BUNDLE = {  # 计算并保存当前步骤的中间状态。
    "model": calibrated_model,  # 执行当前语句以推进本节示例。
    "coverage_model": logistic_model,  # 执行当前语句以推进本节示例。
    **bundle_manifest,  # 执行当前语句以推进本节示例。
    "bundle_fingerprint": bundle_fingerprint,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

REQUIRED_FIELDS = {"request_id", "doc_id", "tenant", "event_time", "text"}  # 计算并保存当前步骤的中间状态。

def parse_zoned_iso8601(value: object, field_path: str) -> datetime:  # 定义本节可复用的核心函数。
    if not isinstance(value, str):  # 按当前条件选择后续控制路径。
        raise ValueError(f"{field_path} 必须是 ISO 8601 字符串")  # 遇到非法合同立即显式失败。
    normalized = value[:-1] + "+00:00" if value.endswith("Z") else value  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        parsed = datetime.fromisoformat(normalized)  # 计算并保存当前步骤的中间状态。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        raise ValueError(f"{field_path} 不是合法 ISO 8601 时间") from exc  # 遇到非法合同立即显式失败。
    if parsed.tzinfo is None or parsed.utcoffset() is None:  # 按当前条件选择后续控制路径。
        raise ValueError(f"{field_path} 必须包含时区偏移或 Z")  # 遇到非法合同立即显式失败。
    return parsed  # 返回当前分支计算出的结果。

def predict_batch(records: list[dict], bundle: dict, max_batch_size: int = 64) -> list[dict]:  # 定义本节可复用的核心函数。
    if not 1 <= len(records) <= max_batch_size:  # 按当前条件选择后续控制路径。
        raise ValueError(f"batch 大小必须在 1..{max_batch_size}")  # 遇到非法合同立即显式失败。
    for index, record in enumerate(records):  # 遍历输入元素以累积或检查结果。
        missing = REQUIRED_FIELDS - record.keys()  # 计算并保存当前步骤的中间状态。
        if missing:  # 按当前条件选择后续控制路径。
            raise ValueError(f"records[{index}] 缺少字段: {sorted(missing)}")  # 遇到非法合同立即显式失败。
        if not isinstance(record["text"], str) or not record["text"].strip():  # 按当前条件选择后续控制路径。
            raise ValueError(f"records[{index}].text 必须是非空字符串")  # 遇到非法合同立即显式失败。
        if record["tenant"] != bundle["serving_tenant"]:  # 按当前条件选择后续控制路径。
            raise ValueError(f"records[{index}].tenant 未路由到当前 bundle")  # 遇到非法合同立即显式失败。
        parse_zoned_iso8601(record["event_time"], f"records[{index}].event_time")  # 执行当前语句以推进本节示例。
    texts = [clean_text(record["text"]) for record in records]  # 计算并保存当前步骤的中间状态。
    probabilities = bundle["model"].predict_proba(texts)  # 计算并保存当前步骤的中间状态。
    classes = bundle["model"].classes_  # 计算并保存当前步骤的中间状态。
    outputs = []  # 计算并保存当前步骤的中间状态。
    for record, text, row in zip(records, texts, probabilities):  # 遍历输入元素以累积或检查结果。
        confidence = float(row.max())  # 计算并保存当前步骤的中间状态。
        coverage = lexical_coverage(text, bundle["coverage_model"])  # 计算并保存当前步骤的中间状态。
        reason = None  # 计算并保存当前步骤的中间状态。
        if coverage < 0.15:  # 按当前条件选择后续控制路径。
            reason = "low_lexical_coverage"  # 计算并保存当前步骤的中间状态。
        elif confidence < bundle["threshold"]:  # 按当前条件选择后续控制路径。
            reason = "low_confidence"  # 计算并保存当前步骤的中间状态。
        outputs.append({  # 执行当前语句以推进本节示例。
            "request_id": record["request_id"], "doc_id": record["doc_id"], "tenant": record["tenant"],  # 执行当前语句以推进本节示例。
            "prediction": "__REJECT__" if reason else classes[row.argmax()],  # 执行当前语句以推进本节示例。
            "confidence": confidence, "probabilities": dict(zip(classes, map(float, row))),  # 执行当前语句以推进本节示例。
            "rejection_reason": reason,  # 执行当前语句以推进本节示例。
            "model_version": bundle["model_version"],  # 执行当前语句以推进本节示例。
            "label_schema_version": bundle["label_schema_version"],  # 执行当前语句以推进本节示例。
            "preprocess_version": bundle["preprocess_version"],  # 执行当前语句以推进本节示例。
            "feature_space_hashes": bundle["feature_space_hashes"],  # 执行当前语句以推进本节示例。
            "bundle_fingerprint": bundle["bundle_fingerprint"],  # 执行当前语句以推进本节示例。
        })  # 执行当前语句以推进本节示例。
    return outputs  # 返回当前分支计算出的结果。

sample_batch = [  # 计算并保存当前步骤的中间状态。
    {"request_id": "R-1", "doc_id": "LIVE-1", "tenant": "tenant-a", "event_time": "2026-07-27T10:00:00+08:00", "text": "物流一直没有更新，包裹在哪里"},  # 执行当前语句以推进本节示例。
    {"request_id": "R-2", "doc_id": "LIVE-2", "tenant": "tenant-a", "event_time": "2026-07-27T10:00:01+08:00", "text": "给我讲讲宇宙学"},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
pd.DataFrame(predict_batch(sample_batch, MODEL_BUNDLE))[["doc_id", "prediction", "confidence", "rejection_reason", "model_version"]]  # 执行当前语句以推进本节示例。


## 10. 最小回归测试与上线门槛

测试不应只断言“代码能跑”。分类服务至少要覆盖：schema 与层级一致性、时间/客户/精确文本隔离、训练/推理清洗一致、概率和为 1、输出顺序、领域外拒识、tenant 路由、带时区时间、各校准折特征空间和 bundle fingerprint，以及固定 golden examples 的预测不发生未解释漂移。线上还需 shadow/canary、延迟和内存压测、回滚演练与人工复核队列。


In [ ]:
assert df.doc_id.is_unique  # 用受控断言验证关键不变量。
assert set(df.label_l2) == set(LABEL_SCHEMA)  # 用受控断言验证关键不变量。
assert train_df.created_at.max() < valid_df.created_at.min() < test_df.created_at.min()  # 用受控断言验证关键不变量。
assert all(left.isdisjoint(right) for i, left in enumerate(time_split_customer_sets) for right in time_split_customer_sets[i + 1:])  # 用受控断言验证关键不变量。
assert all(left.isdisjoint(right) for i, left in enumerate(clean_text_sets) for right in clean_text_sets[i + 1:])  # 用受控断言验证关键不变量。
assert set(entity_train.customer_id).isdisjoint(entity_test.customer_id)  # 用受控断言验证关键不变量。
assert logistic_model.named_steps["tfidf"].vocabulary_  # 用受控断言验证关键不变量。
assert len(feature_space_hashes) == len(calibrated_model.calibrated_classifiers_) == 3  # 用受控断言验证关键不变量。
assert all(len(space_hash) == 64 for space_hash in feature_space_hashes)  # 用受控断言验证关键不变量。
assert len(MODEL_BUNDLE["bundle_fingerprint"]) == 64  # 用受控断言验证关键不变量。

batch_output = predict_batch(sample_batch, MODEL_BUNDLE)  # 计算并保存当前步骤的中间状态。
assert [row["doc_id"] for row in batch_output] == ["LIVE-1", "LIVE-2"]  # 用受控断言验证关键不变量。
assert all(abs(sum(row["probabilities"].values()) - 1.0) < 1e-9 for row in batch_output)  # 用受控断言验证关键不变量。
assert batch_output[1]["prediction"] == "__REJECT__"  # 用受控断言验证关键不变量。
assert all(row["model_version"] == MODEL_BUNDLE["model_version"] for row in batch_output)  # 用受控断言验证关键不变量。
assert all(row["bundle_fingerprint"] == MODEL_BUNDLE["bundle_fingerprint"] for row in batch_output)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    predict_batch([{**sample_batch[0], "text": ""}], MODEL_BUNDLE)  # 执行当前语句以推进本节示例。
    raise AssertionError("空文本本应失败")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

invalid_contract_cases = [  # 计算并保存当前步骤的中间状态。
    ({**sample_batch[0], "tenant": "tenant-b"}, "tenant"),  # 执行当前语句以推进本节示例。
    ({**sample_batch[0], "event_time": "2026-07-27T10:00:00"}, "时区"),  # 执行当前语句以推进本节示例。
    ({**sample_batch[0], "event_time": "not-a-time"}, "ISO 8601"),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
for invalid_record, expected_message in invalid_contract_cases:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        predict_batch([invalid_record], MODEL_BUNDLE)  # 执行当前语句以推进本节示例。
        raise AssertionError(f"非法接口输入本应失败: {expected_message}")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        assert expected_message in str(exc)  # 用受控断言验证关键不变量。

print("所有数据、概率、拒识、版本和接口断言通过。")  # 执行当前语句以推进本节示例。


## 11. 从教学实现走向生产

上线前还要补齐：由标注指南驱动的真实数据集与双人仲裁、按时间和实体隔离的盲测集、近重复检测、多语言/超长文档策略、按成本选阈值、OOD 专项集、可解释的错误样本库、模型注册表、特征/标签漂移告警、灰度与回滚。多标签任务需改为每标签独立概率和阈值；层级任务需报告每层指标与层级一致性。

本 Notebook 的完美或接近完美结果主要来自五类模板词汇高度可分；真实工单会包含省略、跨意图、错别字、新产品、标签政策变化和标注分歧，性能通常显著更低。

### 参考资料（论文与官方文档）

- Pedregosa et al., [Scikit-learn: Machine Learning in Python](https://jmlr.org/papers/v12/pedregosa11a.html), JMLR 2011。
- scikit-learn 官方文档：[TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)、[LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)、[概率校准](https://scikit-learn.org/stable/modules/calibration.html)、[分类指标](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)。
- Niculescu-Mizil & Caruana, [Predicting Good Probabilities with Supervised Learning](https://www.cs.cornell.edu/~alexn/papers/calibration.icml05.crc.rev3.pdf), ICML 2005。
- Ovadia et al., [Can You Trust Your Model's Uncertainty?](https://proceedings.neurips.cc/paper/2019/hash/8558cb408c1d76621371888657d2eb1d-Abstract.html), NeurIPS 2019。

> 教学边界：这里没有声称字符 TF-IDF 是中文分类的最终方案，也没有用这 120 条合成数据估计真实上线收益。它提供的是一条可审计、可替换模型、可持续评估的最小工程骨架。
